# Notebook 6: LoRA Fine-Tuning Qwen2.5-0.5B

Notebook này thực hiện fine-tune Qwen2.5-0.5B (Decoder-only LLM) cho bài toán Text Summarization sử dụng LoRA.
Mục đích: So sánh kiến trúc Decoder-only (Qwen) với Seq2Seq (BARTpho) có cùng số lượng tham số (~0.4B - 0.5B).

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd

## 1. Load Data
Chỉ load tập train và validation từ file CSV đã chuẩn bị ở Notebook 1.

In [ ]:
train_df = pd.read_csv("train_10k.csv")
val_df = pd.read_csv("val_1k.csv")
test_df = pd.read_csv("test_1k.csv")

from datasets import Dataset, DatasetDict
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})
dataset

## 2. Tokenizer & Model
Sử dụng Qwen2.5-0.5B-Instruct hoặc model gốc. Vì là Causal LM, prompt phải có dạng:
`Bài báo: {article}\nTóm tắt: {abstract}<|endoftext|>`

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# Thêm pad token nếu Qwen chưa có
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16, # FP16 cho T4
    device_map="auto",
    trust_remote_code=True
)

## 3. Prepare Dataset for Causal LM

In [ ]:
def format_prompt(example):
    # Định dạng Prompt cho mô hình sinh
    text = f"Tóm tắt bài báo sau:\n{example['article']}\n\nTóm tắt:\n{example['abstract']}{tokenizer.eos_token}"
    return {"text": text}

formatted_dataset = dataset.map(format_prompt)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512, # Limit length for T4 memory
        padding="max_length"
    )

tokenized_datasets = formatted_dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names + ["text"])

# Data collator cho Causal LM (nhãn dịch sang phải 1 token)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 4. Cấu hình LoRA

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] # Qwen linear layers
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_lora_results",
    evaluation_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=2, # Giảm batch_size vì Causal LM ngốn VRAM hơn
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    save_strategy="epoch",
    fp16=True, # Tối ưu T4 GPU
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

# Bắt đầu huấn luyện
trainer.train()

## 6. Lưu mô hình (Adapter)

In [ ]:
trainer.save_model("./qwen-lora-vietnews")
tokenizer.save_pretrained("./qwen-lora-vietnews")
print("Đã lưu LoRA adapter!")